# 05 — Feature matrix (user × item)

Build user_features.parquet + item_features.parquet, join với candidates → full train matrix.
NA giữ nguyên (LightGBM xử lý native).

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local mode: no GCS egress guardrail needed.
print("Local kernel: reading from CACHE_DIR.")

In [ ]:
import time
import pandas as pd
import numpy as np

from utils.features import build_user_features, build_item_features, add_cross_features

t0 = time.time()
train_pos = pd.read_parquet(f"{CACHE_DIR}/train_pos.parquet")
train_pos["event_ts"] = pd.to_datetime(train_pos["event_ts"])
enr = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet")
print(f"train_pos: {len(train_pos):,} | enr: {len(enr):,} | {time.time()-t0:.1f}s")

In [ ]:
import os
pv_path = f"{CACHE_DIR}/events_pageview_30d.parquet"
if os.path.exists(pv_path):
    pv = pd.read_parquet(pv_path)
    pv["event_ts"] = pd.to_datetime(pv["event_ts"])
else:
    pv = pd.DataFrame(columns=["user_id", "item_id", "event_ts", "dwell_time_sec"])
print(f"pageview: {len(pv):,}")

In [ ]:
CUTOFF = pd.Timestamp(VALID_START)
t0 = time.time()
user_feats = build_user_features(train_pos, pv, cutoff_ts=CUTOFF, dim_enriched=enr)
print(f"user_feats: {user_feats.shape} | {time.time()-t0:.1f}s")
new_user_cols = [c for c in user_feats.columns if c.startswith("u_price")]
print(f"  super user features: {new_user_cols}")
user_feats.to_parquet(f"{CACHE_DIR}/user_features.parquet", index=False)

In [ ]:
snap = pd.read_parquet(f"{CACHE_DIR}/snapshot_60d.parquet")
t0 = time.time()
item_feats = build_item_features(train_pos, snap, enr, cutoff_ts=CUTOFF)
print(f"item_feats: {item_feats.shape} | {time.time()-t0:.1f}s")
new_item_cols = [c for c in item_feats.columns if any(k in c for k in ["velocity", "deviation"])]
print(f"  super item features: {new_item_cols}")
item_feats.to_parquet(f"{CACHE_DIR}/item_features.parquet", index=False)

In [ ]:
# Join candidates with features
t0 = time.time()
cands = pd.read_parquet(f"{CACHE_DIR}/candidates_valid.parquet")
full = add_cross_features(cands, user_feats, item_feats)
print(f"full matrix: {full.shape} | {time.time()-t0:.1f}s")

# Attach label from valid_gt
valid_gt = pd.read_parquet(f"{CACHE_DIR}/valid_gt.parquet")
valid_gt["label"] = 1
full = full.merge(valid_gt, on=["user_id", "item_id"], how="left")
full["label"] = full["label"].fillna(0).astype("int8")
print(f"positive labels: {full['label'].sum():,} / {len(full):,} "
      f"({full['label'].mean()*100:.3f}%)")
full.to_parquet(f"{CACHE_DIR}/train_matrix_valid.parquet", index=False)